# Week 1 fantasy range projections — guided walkthrough

This notebook is the short path through the project. It builds a **preseason** player prior from completed seasons only, then simulates the football stat process instead of predicting fantasy points directly.

**Flow:** historical player games → team opportunity → player share/efficiency priors → simulated targets/carries/yards/TDs → fantasy-point distribution → calibration check.

Run the cells in order. The first run downloads the public nflverse weekly player files if `data/raw/player_games.csv` is absent.

## 1. Imports and load the historical player-game table

`normalize_player_games` is the explicit data boundary. It keeps a single row per player-game, derives team targets/rushes/pass attempts **before** excluding QBs, and then retains only RB/WR/TE outputs. The resulting fields such as `team_targets` are realized historical labels; they are never used as same-game features at prediction time.

In [ ]:
from pathlib import Path
import sys

# Support running directly from a clone before an editable install.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fantasy_ranges.data import (
    fetch_player_stats, fetch_roster, fetch_schedule, normalize_player_games,
    week1_candidates_from_roster, week1_matchups_from_schedule,
)
from fantasy_ranges.features import build_preseason_features
from fantasy_ranges.baselines import PositionNormalBaseline, UsageNearestNeighbors, expected_points
from fantasy_ranges.simulation import ComponentSimulator
from fantasy_ranges.backtest import calibration_plot, run_week1_backtest
from fantasy_ranges.projections import project_week1

DATA_PATH = ROOT / 'data/raw/player_games.csv'
if not DATA_PATH.exists():
    # Requires internet only on the first run. CSV ingestion does not require pyarrow.
    games = fetch_player_stats(range(2021, 2026), DATA_PATH.parent)
else:
    games = normalize_player_games(pd.read_csv(DATA_PATH, low_memory=False))

games.shape, games[['season', 'week', 'position']].head()

In [ ]:
games

## 2. The core aggregation

For every historical team-game, we aggregate all player rows to get the pool of opportunities. A receiver's game-level target share is `targets / team_targets`; a rusher's rush share is `carries / team_rush_attempts`. The model treats these shares as uncertain latent quantities in the future.

In [ ]:
team_game = (games.groupby(['season', 'week', 'team'], as_index=False)
    .agg(team_targets=('targets', 'sum'),
         team_carries=('carries', 'sum'),
         team_pass_attempts=('team_pass_attempts', 'first'),
         rb_wr_te_points=('fantasy_points_ppr', 'sum')))
team_game.head()

# The same calculations already live in `normalize_player_games`; this makes them visible.
games[['player_name', 'position', 'targets', 'team_targets', 'target_share_game',
       'carries', 'team_rush_attempts', 'rush_share_game']].head(10)

## 3. Build a Week 1 prior without leakage

Pretend we are about to project the 2025 opener. The candidate list supplies identity/team/position only. `build_preseason_features` filters all history to `season < 2025`, applies recency weighting, and shrinks players with little evidence toward a positional average.

In production, replace `candidates` with a curated 2026 roles file containing roster, rookie/team-change, injury, and expected team-volume inputs. Historical Week 1 rows are used here only because they give us a labeled backtest universe.

In [ ]:
TARGET_SEASON = 2025
week1_candidates = games.loc[(games.season == TARGET_SEASON) & (games.week == 1),
    ['player_id', 'player_name', 'position', 'team']].drop_duplicates()
priors = build_preseason_features(games, week1_candidates, TARGET_SEASON)

prior_columns = ['player_name', 'position', 'team', 'games_sample', 'prior_target_share',
                 'prior_rush_share', 'prior_catch_rate', 'expected_team_pass_attempts',
                 'role_uncertainty', 'rookie', 'team_change']
priors.sort_values('prior_target_share', ascending=False)[prior_columns].head(12)

### What role uncertainty means

It is not an arbitrary fantasy-points standard deviation. It increases when a player has a small sample, is a rookie, changed teams, or had unstable share. In the simulator it **lowers the concentration** of the target-share/carry-share beta distribution. That means uncertainty enters before opportunities and naturally propagates into yardage and touchdowns.

In [ ]:
# Inspect a stable-looking prior and a higher-uncertainty prior from this historical candidate set.
priors.sort_values(['role_uncertainty', 'prior_target_share'], ascending=[True, False])[prior_columns].head(5), \
priors.sort_values('role_uncertainty', ascending=False)[prior_columns].head(5)

## 4. A transparent point-center baseline

Before simulating, compute the center implied by prior opportunities and efficiency:

`expected targets = expected team pass attempts × 0.64 × prior target share`

The baseline then adds receptions, yards, and strongly-shrunk touchdown rates. It is useful as a sanity check, not the final answer.

In [ ]:
priors = priors.copy()
priors['baseline_center'] = expected_points(priors)
priors[['player_name', 'position', 'baseline_center', 'role_uncertainty']].sort_values(
    'baseline_center', ascending=False).head(15)

## 5. Component Monte Carlo simulation

For each draw, the model samples:

1. team pass and rush volume;
2. player target and carry shares, widened when the role is uncertain;
3. discrete targets and carries;
4. receptions conditional on targets;
5. positive, right skewed receiving and rushing yards;
6. receiving and rushing touchdowns with pooled touchdown rates; and
7. PPR fantasy points, followed by the known Week 1 matchup adjustment.

The `fit` call estimates positional priors from the completed history only. Choose one player dynamically so this notebook remains usable even if names/teams change.

In [ ]:
model = ComponentSimulator(simulations=20_000, seed=42).fit(games.loc[games.season < TARGET_SEASON])
player = priors.sort_values('baseline_center', ascending=False).iloc[0]
distribution = model.simulate(player)
summary = distribution.summary()
pd.Series({'player': player.player_name, 'position': player.position} | summary)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(distribution.samples, bins=np.arange(0, 50.5, .5), density=True, color='#3b82f6', alpha=.75)
for label, q, color in [('P10', .1, '#ef4444'), ('Median', .5, '#111827'), ('P90', .9, '#10b981')]:
    value = np.quantile(distribution.samples, q)
    ax.axvline(value, color=color, lw=2, label=f'{label}: {value:.1f}')
ax.set(title=f"{player.player_name}: simulated PPR distribution", xlabel='PPR fantasy points', ylabel='Density')
ax.legend(frameon=False)
plt.show()

## 6. Same median, different range

To isolate role uncertainty, we anchor two WRs to the same 16-point center. The first row retains an established player's history; the second is a rookie and therefore uses a positional prior with low beta concentration. This is the product behavior the project is designed to surface.

In [ ]:
veteran = priors.loc[priors.position.eq('WR')].sort_values('games_sample', ascending=False).iloc[0].copy()
rookie = veteran.copy()
rookie['player_id'], rookie['player_name'], rookie['games_sample'] = 'walkthrough_rookie', 'Illustrative Rookie WR', 0
rookie['rookie'], rookie['team_change'], rookie['role_uncertainty'] = 1, 1, .92
for row in (veteran, rookie):
    row['expected_fantasy_points'] = 16.0  # same center; shape still comes from component model
veteran['role_uncertainty'] = min(veteran['role_uncertainty'], .15)
draws = {'Established WR': model.simulate(veteran, seed_offset=1).samples,
         'Rookie WR': model.simulate(rookie, seed_offset=2).samples}
pd.DataFrame([{ 'profile': name, 'P10': np.quantile(values, .1), 'Median': np.median(values),
                'P90': np.quantile(values, .9), '80% width': np.quantile(values, .9)-np.quantile(values, .1)}
              for name, values in draws.items()])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, values in draws.items():
    ax.hist(values, bins=np.arange(0, 45.5, .5), density=True, histtype='step', lw=2, label=name)
ax.set(title='Equal center, different role uncertainty', xlabel='PPR fantasy points', ylabel='Density')
ax.legend(frameon=False)
plt.show()

## 7. Validate ranges, not just point estimates

This walk-forward backtest holds out each Week 1 from 2022–25. For each season, all player priors use only prior seasons. It compares the raw component model with a position-normal and empirical-neighbor baseline. The conformal version calibrates component quantiles using **previous** Week 1 forecast errors, never the current holdout.

Use 3,000 draws here for notebook speed. The published Week 1 projections use 20,000 draws per player.

In [ ]:
from IPython.display import Image, display
predictions, metrics = run_week1_backtest(games, seasons=range(2022, 2026), simulations=3_000)
display(metrics[['model', 'n', 'median_ae', 'coverage_50', 'width_50', 'coverage_80', 'width_80']])

CALIBRATION_PATH = ROOT / 'outputs/backtest/notebook_calibration.png'
CALIBRATION_PATH.parent.mkdir(parents=True, exist_ok=True)
calibration_plot(predictions, CALIBRATION_PATH)
display(Image(filename=str(CALIBRATION_PATH)))

## 8. Reading the result and next steps

- Prefer the **component MC + walk-forward conformal** model when its coverage is closest to 50% and 80% without an unreasonable width penalty.
- Do not call the raw simulator calibrated if its coverage misses nominal coverage; the backtest is meant to catch exactly that.
- For 2026, create a versioned candidate/roles input instead of using historical Week 1 players. Include team, rookie/team-change flags, expected team volume, and optional consensus center.
- Routes, snaps, red-zone usage, injury status, QB/coaching changes, and external consensus are valuable feature additions, but they must be lagged/as-of dated and should widen priors when uncertain.

The reusable implementation lives in `src/fantasy_ranges/`; this notebook deliberately exposes the minimal conceptual path rather than duplicating the package.

## 9. Actual 2026 Week 1 projections

The earlier rookie comparison was only a teaching aid. This final section builds the real 2026 active RB/WR/TE candidate list from the Week 1 roster snapshot and attaches the known Week 1 opponent from the schedule. It then uses completed 2021–25 player-game data, 20,000 simulations per player, the matchup adjustment, and walk-forward conformal calibration.

In [ ]:
TARGET_PROJECTION_SEASON = 2026
INPUT_DIR = ROOT / 'data/inputs'
ROSTER_PATH = INPUT_DIR / f'roster_{TARGET_PROJECTION_SEASON}.csv'
SCHEDULE_PATH = INPUT_DIR / 'games.csv'

roster_2026 = pd.read_csv(ROSTER_PATH, low_memory=False) if ROSTER_PATH.exists() else fetch_roster(TARGET_PROJECTION_SEASON, INPUT_DIR)
schedule = pd.read_csv(SCHEDULE_PATH, low_memory=False) if SCHEDULE_PATH.exists() else fetch_schedule(INPUT_DIR)
matchups_2026 = week1_matchups_from_schedule(schedule, TARGET_PROJECTION_SEASON)
candidates_2026 = week1_candidates_from_roster(roster_2026, TARGET_PROJECTION_SEASON, matchups=matchups_2026)
features_2026 = build_preseason_features(games, candidates_2026, TARGET_PROJECTION_SEASON)

feature_columns_2026 = ['player_name', 'position', 'team', 'opponent', 'expected_team_pass_attempts',
                        'expected_team_rush_attempts', 'prior_target_share', 'prior_rush_share',
                        'prior_catch_rate', 'role_uncertainty', 'matchup_multiplier']
features_2026[feature_columns_2026].sort_values('prior_target_share', ascending=False).head(12)

In [ ]:
PROJECTION_PATH = ROOT / 'outputs/week1_2026/projections.csv'
if PROJECTION_PATH.exists():
    week1_2026 = pd.read_csv(PROJECTION_PATH)
else:
    week1_2026 = project_week1(games, candidates_2026, TARGET_PROJECTION_SEASON, simulations=20_000)
    PROJECTION_PATH.parent.mkdir(parents=True, exist_ok=True)
    week1_2026.to_csv(PROJECTION_PATH, index=False)

projection_columns = ['player_name', 'position', 'team', 'opponent', 'matchup_multiplier', 'mean',
                      'p10', 'p25', 'p50', 'p75', 'p90', 'role_uncertainty', 'uncertainty', 'calibration']
week1_2026[projection_columns].sort_values('mean', ascending=False).head(30)